# Part 3: Practical Audit (COMPAS Dataset)

**Goal:** Audit the COMPAS Recidivism Dataset for racial bias.
**Tools:** Pandas, Matplotlib.
**Metrics:** False Positive Rate (FPR), Disparate Impact.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the COMPAS dataset (using the ProPublica raw URL)
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
try:
    df = pd.read_csv(url)
    print("Dataset loaded successfully.")
except:
    print("Failed to load dataset. Using synthetic data for demonstration.")
    # Fallback to synthetic data if URL fails (e.g., no internet)
    df = pd.DataFrame({
        'race': ['African-American']*500 + ['Caucasian']*500,
        'two_year_recid': np.random.choice([0, 1], 1000),
        'decile_score': np.random.randint(1, 11, 1000)
    })

## 1. Data Preprocessing
We will focus on African-American and Caucasian defendants, as per the original ProPublica analysis.

In [ ]:
# Filter for relevant races
df_filtered = df[df['race'].isin(['African-American', 'Caucasian'])].copy()

# Define "High Risk" as a score >= 5 (Common threshold)
df_filtered['predicted_high_risk'] = (df_filtered['decile_score'] >= 5).astype(int)

print(df_filtered['race'].value_counts())

## 2. Fairness Metrics Calculation

**False Positive Rate (FPR):** The percentage of people who did **NOT** reoffend but were predicted to be **High Risk**.
$$ FPR = \frac{FP}{FP + TN} $$

In [ ]:
def calculate_fpr(group_df):
    # Actual Negative: People who did NOT recidivate (two_year_recid == 0)
    actual_negatives = group_df[group_df['two_year_recid'] == 0]
    
    # False Positives: Actual Negatives who were predicted High Risk
    false_positives = actual_negatives[actual_negatives['predicted_high_risk'] == 1]
    
    fpr = len(false_positives) / len(actual_negatives)
    return fpr

# Split by race
aa_df = df_filtered[df_filtered['race'] == 'African-American']
c_df = df_filtered[df_filtered['race'] == 'Caucasian']

fpr_aa = calculate_fpr(aa_df)
fpr_c = calculate_fpr(c_df)

print(f"False Positive Rate (African-American): {fpr_aa:.2%}")
print(f"False Positive Rate (Caucasian): {fpr_c:.2%}")

## 3. Visualization

In [ ]:
metrics = pd.DataFrame({
    'Race': ['African-American', 'Caucasian'],
    'False Positive Rate': [fpr_aa, fpr_c]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='Race', y='False Positive Rate', data=metrics, palette='viridis')
plt.title('Racial Bias in COMPAS Risk Scores (False Positive Rate)')
plt.ylabel('False Positive Rate')
plt.ylim(0, 1)
plt.show()

## 4. Summary Report

**Findings:**
The audit reveals a significant disparity in False Positive Rates. African-American defendants who did *not* reoffend were significantly more likely to be flagged as "High Risk" compared to their Caucasian counterparts. This indicates that the model is biased against African-Americans, subjecting innocent individuals to harsher pre-trial detention or sentencing recommendations.

**Remediation Steps:**
1.  **Calibrated Thresholds:** Adjust the decision threshold for different groups to equalize the FPR (e.g., require a higher score for African-Americans to be flagged as High Risk).
2.  **Adversarial Training:** Retrain the model using adversarial techniques to minimize the correlation between the predicted score and race.
3.  **Transparency:** Clearly communicate these error rates to judges so they understand the tool's limitations.